In [46]:
import pandas as pd

In [47]:
df_01 = pd.read_csv(
    "C:/Users/user/team01/부산항만공사_물동량 예측_20250331.csv",
    encoding="cp949"
)
df_01.head(3)

,연도,월,연월,수출입구분,수출입구분설명,물동량
0,2003,2,2003-02,OT,수출환적,"161,074"
1,2003,11,2003-11,IT,수입환적,"170,734"
2,2006,1,2006-01,OT,수출환적,"203,348"


In [48]:
df_vol = df_01.copy()

In [50]:
df_vol_year = df_vol.groupby(["연도", "월"])["물동량"].sum().reset_index()
df_vol_year_case = df_vol.groupby(["연도", "월","수출입구분설명"])["물동량"].sum().reset_index()

In [51]:
print("=== df_vol_year ===")
print(df_vol_year.head(3))
print("\n=== df_vol_year_case ===")
print(df_vol_year_case.head(3))

=== df_vol_year ===
     연도  월     물동량
0  1992  1 196,572
1  1992  2 195,572
2  1992  3 242,358

=== df_vol_year_case ===
     연도  월 수출입구분설명     물동량
0  1992  1      수입  92,005
1  1992  1      수출 104,567
2  1992  2      수입  91,669


In [52]:
# 계절지수
# 모든 연도의 같은 월 평균
monthly_mean = df_vol_year.groupby("월")["물동량"].mean()
# 월별 계절지수
seasonal_index = monthly_mean / monthly_mean.mean()
# 실제로 매년 반복되는지도 확인
year_month = df_vol_year.pivot_table(
    index="연도",
    columns="월",
    values="물동량",
    aggfunc="mean"
)
year_month.head(3)

월,1,2,3,4,5,6,7,8,9,10,11,12
연도,,,,,,,,,,,,
1992,"196,572","195,572","242,358","229,614","222,359","219,336","225,557","213,642","213,136","221,729","224,744","230,755"
1993,"201,356","208,964","256,119","239,664","259,552","242,137","248,903","252,676","263,234","245,653","246,546","276,431"
1994,"284,910","252,518","295,176","295,324","295,170","290,340","314,422","292,576","295,645","313,063","315,149","338,224"


In [53]:
monthly_diff = monthly_mean.diff()
monthly_diff

월
1        NaN
2    -75,875
3    131,942
4    -18,021
5     13,461
6    -34,420
7     26,006
8    -36,209
9    -23,120
10    51,770
11   -11,518
12    13,808
Name: 물동량, dtype: float64

In [54]:
# 0초과 3이하
bins = [0, 3, 6, 9, 12]
labels = ["1분기", "2분기", "3분기", "4분기"]

df_vol_year["분기별"] = pd.cut(
    df_vol_year["월"],
    bins=bins,
    labels=labels,
)

df_vol_year_case["분기별"] = pd.cut(
    df_vol_year_case["월"],
    bins=bins,
    labels=labels,
)

print("=== df_vol_year ===")
print(df_vol_year.head())
print("\n=== df_vol_year_case ===")
print(df_vol_year_case.head())

=== df_vol_year ===
     연도  월     물동량  분기별
0  1992  1 196,572  1분기
1  1992  2 195,572  1분기
2  1992  3 242,358  1분기
3  1992  4 229,614  2분기
4  1992  5 222,359  2분기

=== df_vol_year_case ===
     연도  월 수출입구분설명     물동량  분기별
0  1992  1      수입  92,005  1분기
1  1992  1      수출 104,567  1분기
2  1992  2      수입  91,669  1분기
3  1992  2      수출 103,903  1분기
4  1992  3      수입 105,276  1분기


In [55]:
df_vol_quarter = df_vol_year.groupby(["분기별"])["물동량"].sum().reset_index()
df_vol_quarter_case = df_vol_year_case.groupby(["분기별", "수출입구분설명"])["물동량"].sum().reset_index()

print("=== df_vol_quarter ===")
print(df_vol_quarter.head())
print("\n=== df_vol_quarter_case ===")
print(df_vol_quarter_case.head(15))

=== df_vol_quarter ===
   분기별         물동량
0  1분기 110,874,627
1  2분기 112,913,309
2  3분기 110,507,657
3  4분기 112,607,506

=== df_vol_quarter_case ===
    분기별 수출입구분설명        물동량
0   1분기      수입 30,668,165
1   1분기    수입환적 25,158,125
2   1분기      수출 30,141,114
3   1분기    수출환적 24,907,222
4   2분기      수입 31,492,748
5   2분기    수입환적 25,423,383
6   2분기      수출 30,794,907
7   2분기    수출환적 25,202,270
8   3분기      수입 30,556,567
9   3분기    수입환적 25,132,334
10  3분기      수출 30,121,032
11  3분기    수출환적 24,697,724
12  4분기      수입 31,666,150
13  4분기    수입환적 25,383,192
14  4분기      수출 30,644,794


In [59]:
df_vol_quarter_case.groupby(["분기별", "수출입구분설명"])["물동량"].sum()
df_vol_quarter_case

,분기별,수출입구분설명,물동량
0,1분기,수입,"30,668,165"
1,1분기,수입환적,"25,158,125"
2,1분기,수출,"30,141,114"
3,1분기,수출환적,"24,907,222"
4,2분기,수입,"31,492,748"
5,2분기,수입환적,"25,423,383"
6,2분기,수출,"30,794,907"
7,2분기,수출환적,"25,202,270"
8,3분기,수입,"30,556,567"
9,3분기,수입환적,"25,132,334"


In [60]:
# 모든 분기에서 수입 물동량이 가장 크고, 수출환적 물동량이 가장 작다
df_vol_quarter_pv = df_vol_quarter_case.pivot_table(
    index="분기별",
    columns="수출입구분설명",
    values="물동량",
)
df_vol_quarter_pv.head(4)

수출입구분설명,수입,수입환적,수출,수출환적
분기별,,,,
1분기,"30,668,165","25,158,125","30,141,114","24,907,222"
2분기,"31,492,748","25,423,383","30,794,907","25,202,270"
3분기,"30,556,567","25,132,334","30,121,032","24,697,724"
4분기,"31,666,150","25,383,192","30,644,794","24,913,371"


In [68]:
# 연도별 분기별
df_vol_quarter_pv_year = df_vol_year_case.pivot_table(
    index=["연도", "분기별"], columns="수출입구분설명", values="물동량", aggfunc="sum"
)
df_vol_quarter_pv_year = df_vol_quarter_pv_year[
    df_vol_quarter_pv_year.index.get_level_values("연도") >= 2001
]
df_vol_quarter_pv_year

수출입구분설명         수입      수입환적        수출      수출환적
연도   분기별                                        
2001 1분기   608,834   354,068   628,788   332,340
     2분기   660,308   369,984   646,184   336,370
     3분기   630,866   404,379   637,541   370,433
     4분기   671,124   401,610   646,048   373,923
2002 1분기   638,572   431,946   664,868   410,983
...            ...       ...       ...       ...
2024 1분기 1,352,858 1,656,608 1,366,167 1,639,101
     2분기 1,408,204 1,722,986 1,416,706 1,687,279
     3분기 1,333,556 1,675,127 1,356,116 1,672,401
     4분기 1,315,074 1,740,010 1,356,156 1,703,671
2025 1분기   860,448 1,169,006   872,336 1,187,838

[97 rows x 4 columns]

,분기별,수출입구분설명,물동량
0,1분기,수입,"30,668,165"
1,1분기,수입환적,"25,158,125"
2,1분기,수출,"30,141,114"
3,1분기,수출환적,"24,907,222"
4,2분기,수입,"31,492,748"
5,2분기,수입환적,"25,423,383"
6,2분기,수출,"30,794,907"
7,2분기,수출환적,"25,202,270"
8,3분기,수입,"30,556,567"
9,3분기,수입환적,"25,132,334"


In [ ]:
#범주형
bins = [0, 50_000_000, 100_000_000, 150_000_000, float("inf")]
labels = ["50,000,000 이하(낮음)", "100,000,000 이하(보통)", "150,000,000 이하(높음)", "150,000,000 초과(매우 높음)"]

df_vol_quarter["물동량_범주"] = pd.cut(
    df_vol_year["물동량"],
    bins=bins,
    labels=labels
)

df_vol_quarter_case["물동량_범주"] = pd.cut(
    df_vol_year["물동량"],
    bins=bins,
    labels=labels
)

print("=== 분기별 물동량 ===")
print(df_vol_quarter.head())
print("\n=== 분기별 수출입별 물동량 ===")
print(df_vol_quarter_case.head())

=== 분기별 물동량 ===
   분기별         물동량             물동량_범주
0  1분기 110,874,627  50,000,000 이하(낮음)
1  2분기 112,913,309  50,000,000 이하(낮음)
2  3분기 110,507,657  50,000,000 이하(낮음)
3  4분기 112,607,506  50,000,000 이하(낮음)

=== 분기별 수출입별 물동량 ===
   분기별 수출입구분설명        물동량             물동량_범주
0  1분기      수입 30,668,165  50,000,000 이하(낮음)
1  1분기    수입환적 25,158,125  50,000,000 이하(낮음)
2  1분기      수출 30,141,114  50,000,000 이하(낮음)
3  1분기    수출환적 24,907,222  50,000,000 이하(낮음)
4  2분기      수입 31,492,748  50,000,000 이하(낮음)


In [ ]:
df_vol_quarter.groupby("분기별")["물동량"].mean()

분기별
1분기    1.108746e+08
2분기    1.129133e+08
3분기    1.105077e+08
4분기    1.126075e+08
Name: 물동량, dtype: float64

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format
#pd.reset_option("display.float_format")

In [ ]:
test = df_vol_quarter.groupby(
    "분기별", observed=True
)["물동량"].mean()

test.diff().abs()

분기별
1분기         NaN
2분기   2,038,682
3분기   2,405,652
4분기   2,099,850
Name: 물동량, dtype: float64